In [1]:
# Import Libraries
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import classification_report, confusion_matrix
VOCAB_SIZE = 10000
MAX_LEN = 200

In [2]:
(X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)
print('Training samples:', len(X_train))
print('Test samples:', len(X_test))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training samples: 25000
Test samples: 25000


In [3]:
X_train = pad_sequences(X_train, maxlen=MAX_LEN, padding='post', truncating='post')
X_test = pad_sequences(X_test, maxlen=MAX_LEN, padding='post', truncating='post')
print(X_train.shape)

(25000, 200)


In [4]:
model = keras.Sequential([
 layers.Input(shape=(MAX_LEN,)),
 layers.Embedding(input_dim=VOCAB_SIZE, output_dim=128),
 layers.LSTM(64, dropout=0.2, recurrent_dropout=0.2),
 layers.Dense(32, activation='relu'),
 layers.Dropout(0.3),
 layers.Dense(1, activation='sigmoid')
])
model.compile(
 optimizer='adam',
 loss='binary_crossentropy',
 metrics=['accuracy']
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,331,521 (5.08 MB)

 Trainable params: 1,331,521 (5.08 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
history = model.fit(
 X_train, y_train,
 validation_split=0.2,
 epochs=5,
 batch_size=64
)


Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 222s 684ms/step - accuracy: 0.5522 - loss: 0.6791 - val_accuracy: 0.6772 - val_loss: 0.6259
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 213s 682ms/step - accuracy: 0.5520 - loss: 0.6745 - val_accuracy: 0.5132 - val_loss: 0.6927
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 259s 673ms/step - accuracy: 0.5293 - loss: 0.6906 - val_accuracy: 0.5550 - val_loss: 0.6860
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 213s 679ms/step - accuracy: 0.6582 - loss: 0.6149 - val_accuracy: 0.8308 - val_loss: 0.4134
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 265s 688ms/step - accuracy: 0.8624 - loss: 0.3563 - val_accuracy: 0.8682 - val_loss: 0.3230


In [6]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Negative','Positive']))

782/782 ━━━━━━━━━━━━━━━━━━━━ 100s 127ms/step
[[10585  1915]
 [ 1739 10761]]
              precision    recall  f1-score   support

    Negative       0.86      0.85      0.85     12500
    Positive       0.85      0.86      0.85     12500

    accuracy                           0.85     25000
   macro avg       0.85      0.85      0.85     25000
weighted avg       0.85      0.85      0.85     25000

